In [2]:
from datetime import datetime
import meteostat as ms
import pandas as pd
### get charlotte houly weather data from Meteostat and merge it with car crash data 
# Disable Meteostat large request block
ms.config.block_large_requests = False

#use coordinates to locate Charlotte
charlotte_point = ms.Point(35.2271, -80.8431)

# Find the nearest weather station to Charlotte
stations = ms.stations.nearby(charlotte_point, limit=1)
station_id = stations.index[0]
print(f"Found nearest weather station ID: {station_id}")

# select the start and end dates for the hourly weather data
start = datetime(2010, 1, 1)
end = datetime(2025, 12, 31)

print(
    "Get hourly weather data from 2010 to 2025 from Meteostat"

)
data = ms.hourly(station_id, start, end)
df_weather = data.fetch()


df_weather = df_weather.reset_index()

print("Reading crash data...")
df_crashes = pd.read_csv("Crashes.csv")


df_crashes["MILT_TIME_STR"] = (
    df_crashes["MILT_TIME"].astype(str).str.zfill(4)
)
df_crashes["time_formatted"] = (
    df_crashes["MILT_TIME_STR"].str.slice(0, 2)
    + ":"
    + df_crashes["MILT_TIME_STR"].str.slice(2, 4)
)
df_crashes["datetime_str"] = (
    df_crashes["DATE_VAL"].astype(str).str.split(" ").str[0]
    + " "
    + df_crashes["time_formatted"]
)
df_crashes["datetime"] = pd.to_datetime(
    df_crashes["datetime_str"], format="%Y/%m/%d %H:%M", errors="coerce"
)

# merge data by hour
print(" merging by hour...")
df_crashes["crash_hour_key"] = df_crashes["datetime"].dt.floor("h")

df_weather["weather_datetime"] = pd.to_datetime(df_weather["time"])
df_weather["weather_hour_key"] = df_weather["weather_datetime"].dt.floor("h")

# left join two datasets
df_final = pd.merge(
    df_crashes,
    df_weather,
    left_on="crash_hour_key",
    right_on="weather_hour_key",
    how="left",
)


output_path = "charlotte_crashes_with_meteostat.csv"
df_final.to_csv(output_path, index=False)
print(f"Success! Merged file successfully saved to: {output_path}")

Found nearest weather station ID: 72314
Get hourly weather data from 2010 to 2025 from Meteostat
Reading crash data...
 merging by hour...
Success! Merged file successfully saved to: charlotte_crashes_with_meteostat.csv


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

# get the distribution of crash severity (CRSH_LEVL) in the merged dataset
df = pd.read_csv("charlotte_crashes_with_meteostat.csv")

counts = df["CRSH_LEVL"].value_counts().sort_index()
print(counts)

print("\n CRSH_LEVL (%) ---")
percentages = df["CRSH_LEVL"].value_counts(normalize=True).sort_index() * 100
print(percentages.round(2))




CRSH_LEVL
1.0      1124
2.0      1654
3.0     28692
4.0    106023
5.0    403086
6.0         2
Name: count, dtype: int64

 CRSH_LEVL (%) ---
CRSH_LEVL
1.0     0.21
2.0     0.31
3.0     5.31
4.0    19.61
5.0    74.57
6.0     0.00
Name: proportion, dtype: float64


In [4]:
## Drop rows with CRSH_LEVL == 6 due to invalid data
#  The data is not balanced in crash_level, level 1-3 are rare
# create a new binary, combining CRSH_LEVL 1-4 into a single category: with injury or fatality, and 5 as no injury
df = df[df["CRSH_LEVL"] != 6]
## combine CRSH_LEVL 1-4 into a single category: with injury or fatality, and 5 as no injury
df["IS_INJURY"] = df["CRSH_LEVL"].apply(lambda x: 0 if x == 5 else 1)


In [6]:
# Use logistic regression for binary prediction of injury

import numpy as np
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#load in the data
df = pd.read_csv("charlotte_crashes_with_meteostat.csv")

# Binary label: Levels 1-4 merged to 1 (Injury/Severe), Level 5 to 0 (No Injury)
df["IS_INJURY"] = df["CRSH_LEVL"].apply(lambda x: 0 if x == 5 else 1)


# Extract hour
df["Hour"] = df["MILT_TIME"] // 100

# create rush hour flag (e.g., 7-9 AM, 4-7 PM)
df["Is_Rush_Hour"] = df["Hour"].apply(
    lambda h: 1 if (7 <= h <= 9) or (16 <= h <= 19) else 0
)

# Weekend flag (6, 7 represent weekends in DAY_OF_WEEK)
df["Is_Weekend"] = df["DAY_OF_WEEK"].apply(lambda d: 1 if d in [6, 7] else 0)

#define indicator features for has rain (1) or no rain (0)
df["Has_Rain"] = df["prcp"].apply(lambda p: 1 if p > 0 else 0)

# One-Hot Encoding for crash types
df_encoded = pd.get_dummies(df, columns=["CRASH_TYPE"], drop_first=True)

#list candidate feature pool
candidate_features = [
    "LATITUDE",
    "LONGITUDE",
    "DAY_OF_WEEK",
    "Hour",
    "Is_Rush_Hour",
    "Is_Weekend",
    "temp",
    "rhum",
    "Has_Rain",
    "wspd",
    "pres",
    "cldc",
]

# Include One-Hot encoded crash type columns
crash_type_cols = [col for col in df_encoded.columns if "CRASH_TYPE_" in col]
all_features = candidate_features + crash_type_cols

# drop rows with missing values
df_model = df_encoded.dropna(subset=all_features + ["IS_INJURY"]).copy()

X = df_model[all_features]
y = df_model["IS_INJURY"]

# create raining and testing data, can use different test size
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# use RFE to select predictive features in the logistic regression model
estimator = LogisticRegression(class_weight="balanced", random_state=42)
selector = RFE(estimator, n_features_to_select=12, step=1)

selector.fit(X_train_scaled, y_train)

# show features selected by RFE
selected_features = np.array(all_features)[selector.support_]
print("Optimal Feature Subset Selected by RFE:")
for i, feat in enumerate(selected_features, 1):
  print(f"{i}. {feat}")

# Transform datasets with selected features
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

# Model Training & Evaluation 
estimator.fit(X_train_selected, y_train)

y_pred = estimator.predict(X_test_selected)
y_proba = estimator.predict_proba(X_test_selected)[:, 1]

print("\n Detailed Classification Report, note: Default Threshold is 0.5) ")
print(classification_report(y_test, y_pred, zero_division=0))

auc_score = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC Score: {auc_score:.4f}")

Optimal Feature Subset Selected by RFE:
1. CRASH_TYPE_Animal
2. CRASH_TYPE_Backing up
3. CRASH_TYPE_Fixed object
4. CRASH_TYPE_Movable object
5. CRASH_TYPE_Other collision with vehicle*
6. CRASH_TYPE_Parked motor vehicle
7. CRASH_TYPE_Pedalcyclist
8. CRASH_TYPE_Pedestrian
9. CRASH_TYPE_Rear end; slow or stop
10. CRASH_TYPE_Right turn; different roadways
11. CRASH_TYPE_Sideswipe; opposite direction
12. CRASH_TYPE_Sideswipe; same direction

 Detailed Classification Report, note: Default Threshold is 0.5) 
              precision    recall  f1-score   support

           0       0.90      0.45      0.60     58953
           1       0.35      0.86      0.49     20080

    accuracy                           0.55     79033
   macro avg       0.62      0.65      0.54     79033
weighted avg       0.76      0.55      0.57     79033

ROC-AUC Score: 0.6954


In [7]:

#increase number of predictors to 16 since the previous step only select crash type

estimator = LogisticRegression(class_weight="balanced", random_state=42)
selector = RFE(estimator, n_features_to_select=20, step=1)

selector.fit(X_train_scaled, y_train)

# Print selected features
selected_features = np.array(all_features)[selector.support_]
print("Optimal Feature Subset Selected by RFE")
for i, feat in enumerate(selected_features, 1):
  print(f"{i}. {feat}")


X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

estimator.fit(X_train_selected, y_train)

y_pred = estimator.predict(X_test_selected)
y_proba = estimator.predict_proba(X_test_selected)[:, 1]

print("\n Detailed Classification Report (Default Threshold 0.5) ")
print(classification_report(y_test, y_pred, zero_division=0))

auc_score = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC Score: {auc_score:.4f}")

Optimal Feature Subset Selected by RFE
1. CRASH_TYPE_Animal
2. CRASH_TYPE_Backing up
3. CRASH_TYPE_Fixed object
4. CRASH_TYPE_Head On
5. CRASH_TYPE_Left turn; same roadway
6. CRASH_TYPE_Movable object
7. CRASH_TYPE_Other collision with vehicle*
8. CRASH_TYPE_Overturn/rollover
9. CRASH_TYPE_Parked motor vehicle
10. CRASH_TYPE_Pedalcyclist
11. CRASH_TYPE_Pedestrian
12. CRASH_TYPE_Ran off road left
13. CRASH_TYPE_Ran off road right
14. CRASH_TYPE_Rear end; slow or stop
15. CRASH_TYPE_Rear end; turn
16. CRASH_TYPE_Right turn; different roadways
17. CRASH_TYPE_Right turn; same roadway
18. CRASH_TYPE_Sideswipe; opposite direction
19. CRASH_TYPE_Sideswipe; same direction
20. CRASH_TYPE_crossed centerline

 Detailed Classification Report (Default Threshold 0.5) 
              precision    recall  f1-score   support

           0       0.90      0.47      0.61     58953
           1       0.35      0.84      0.49     20080

    accuracy                           0.56     79033
   macro avg     

In [8]:
## still no other predictors selected except crash type
# force two predictors has_rain and is_rush_hour into the model
forced_features = ["Has_Rain", "Is_Rush_Hour"]

# Redefine the candidate pool for RFE, excluding the forced features
remaining_candidates = [
    "LATITUDE",
    "LONGITUDE",
    "DAY_OF_WEEK",
    "Hour",
    "Is_Weekend",
    "temp",
    "rhum",
    "wspd",
    "pres",
    "cldc",
]
crash_type_cols = [col for col in df_encoded.columns if "CRASH_TYPE_" in col]
rfe_pool = remaining_candidates + crash_type_cols


X_rfe_candidates = df_model[rfe_pool]
X_train_rfe, X_test_rfe = train_test_split(
    X_rfe_candidates, test_size=0.2, random_state=42, stratify=df_model["IS_INJURY"]
)


scaler_rfe = StandardScaler()
X_train_rfe_scaled = scaler_rfe.fit_transform(X_train_rfe)
X_test_rfe_scaled = scaler_rfe.transform(X_test_rfe)


estimator = LogisticRegression(class_weight="balanced", random_state=42)
selector = RFE(estimator, n_features_to_select=12, step=1)
selector.fit(X_train_rfe_scaled, y_train)


rfe_selected_features = np.array(rfe_pool)[selector.support_]


final_selected_features = forced_features + list(rfe_selected_features)

print("Final Selected Features (Including Forced Features)")
for i, feat in enumerate(final_selected_features, 1):
  print(f"{i}. {feat}")


X_final = df_model[final_selected_features]
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)


scaler_final = StandardScaler()
X_train_f_scaled = scaler_final.fit_transform(X_train_f)
X_test_f_scaled = scaler_final.transform(X_test_f)


estimator.fit(X_train_f_scaled, y_train_f)
y_pred = estimator.predict(X_test_f_scaled)
y_proba = estimator.predict_proba(X_test_f_scaled)[:, 1]


print("\n final model evaluation with forced features:")
print(classification_report(y_test_f, y_pred, zero_division=0))

auc_score = roc_auc_score(y_test_f, y_proba)
print(f"ROC-AUC Score: {auc_score:.4f}")

Final Selected Features (Including Forced Features)
1. Has_Rain
2. Is_Rush_Hour
3. CRASH_TYPE_Animal
4. CRASH_TYPE_Backing up
5. CRASH_TYPE_Fixed object
6. CRASH_TYPE_Movable object
7. CRASH_TYPE_Other collision with vehicle*
8. CRASH_TYPE_Parked motor vehicle
9. CRASH_TYPE_Pedalcyclist
10. CRASH_TYPE_Pedestrian
11. CRASH_TYPE_Rear end; slow or stop
12. CRASH_TYPE_Right turn; different roadways
13. CRASH_TYPE_Sideswipe; opposite direction
14. CRASH_TYPE_Sideswipe; same direction

 final model evaluation with forced features:
              precision    recall  f1-score   support

           0       0.90      0.45      0.60     58953
           1       0.35      0.86      0.49     20080

    accuracy                           0.55     79033
   macro avg       0.62      0.65      0.54     79033
weighted avg       0.76      0.55      0.57     79033

ROC-AUC Score: 0.6976


In [9]:
## Use random forest to see if we can get better results
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# Load Data & Label Transformation
df = pd.read_csv("charlotte_crashes_with_meteostat.csv")

# Binary label: Levels 1-4 assigned to 1 (Injury/Severe), Level 5 to 0 (No Injury)
df["IS_INJURY"] = df["CRSH_LEVL"].apply(lambda x: 0 if x == 5 else 1)

# Extract hour
df["Hour"] = df["MILT_TIME"] // 100

# rush hour flag (e.g., 7-9 AM, 4-7 PM)
df["Is_Rush_Hour"] = df["Hour"].apply(
    lambda h: 1 if (7 <= h <= 9) or (16 <= h <= 19) else 0
)

# Weekend flag (assuming 6, 7 represent weekends in DAY_OF_WEEK)
df["Is_Weekend"] = df["DAY_OF_WEEK"].apply(lambda d: 1 if d in [6, 7] else 0)

# Has rain flag
df["Has_Rain"] = df["prcp"].apply(lambda p: 1 if p > 0 else 0)

# One-Hot Encoding for crash types
df_encoded = pd.get_dummies(df, columns=["CRASH_TYPE"], drop_first=True)

# Define candidate feature pool (Random forest can handle all features at once)
candidate_features = [
    "LATITUDE",
    "LONGITUDE",
    "DAY_OF_WEEK",
    "Hour",
    "Is_Rush_Hour",
    "Is_Weekend",
    "temp",
    "rhum",
    "Has_Rain",
    "wspd",
    "pres",
    "cldc",
]

# Include One-Hot encoded crash type columns
crash_type_cols = [col for col in df_encoded.columns if "CRASH_TYPE_" in col]
all_features = candidate_features + crash_type_cols

# Drop rows with missing values
df_model = df_encoded.dropna(subset=all_features + ["IS_INJURY"]).copy()

X = df_model[all_features]
y = df_model["IS_INJURY"]

# Train/Test Split 
# Note: random forest do not require feature scaling (StandardScaler)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest Model Training 
# class_weight='balanced' handles the severe class imbalance automatically, the 75/25 is not that imblanced
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,  
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,  
)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)
y_proba = rf_classifier.predict_proba(X_test)[:, 1]

print("\n Random Forest Classification Report ")
print(classification_report(y_test, y_pred, zero_division=0))

auc_score = roc_auc_score(y_test, y_proba)
print(f"Random Forest ROC-AUC Score: {auc_score:.4f}")


# Random Forest automatically calculates feature importance
importances = rf_classifier.feature_importances_
feature_importance_df = pd.DataFrame(
    {"Feature": all_features, "Importance": importances}
).sort_values(by="Importance", ascending=False)

print("\n Top 10 Most Important Features ")
print(feature_importance_df.head(10))


 Random Forest Classification Report 
              precision    recall  f1-score   support

           0       0.90      0.47      0.62     58953
           1       0.35      0.84      0.50     20080

    accuracy                           0.56     79033
   macro avg       0.62      0.66      0.56     79033
weighted avg       0.76      0.56      0.58     79033

Random Forest ROC-AUC Score: 0.7138

 Top 10 Most Important Features 
                                 Feature  Importance
23       CRASH_TYPE_Parked motor vehicle    0.237127
13                 CRASH_TYPE_Backing up    0.120331
25                 CRASH_TYPE_Pedestrian    0.096064
35  CRASH_TYPE_Sideswipe; same direction    0.092311
0                               LATITUDE    0.048250
1                              LONGITUDE    0.043627
18    CRASH_TYPE_Left turn; same roadway    0.042407
10                                  pres    0.036100
6                                   temp    0.032415
7                                 

In [10]:
## choose different hyperparameters for random forest to see if we can get better results
# Add more parameters to the Random Forest Classifier


rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,  
    class_weight="balanced",
    min_samples_split=5,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1,  
)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)
y_proba = rf_classifier.predict_proba(X_test)[:, 1]

print("\n Random Forest Classification Report ")
print(classification_report(y_test, y_pred, zero_division=0))

auc_score = roc_auc_score(y_test, y_proba)
print(f"Random Forest ROC-AUC Score: {auc_score:.4f}")


# Random Forest automatically calculates feature importance
importances = rf_classifier.feature_importances_
feature_importance_df = pd.DataFrame(
    {"Feature": all_features, "Importance": importances}
).sort_values(by="Importance", ascending=False)

print("\n Top 10 Most Important Features ")
print(feature_importance_df.head(10))


 Random Forest Classification Report 
              precision    recall  f1-score   support

           0       0.89      0.48      0.63     58953
           1       0.35      0.83      0.50     20080

    accuracy                           0.57     79033
   macro avg       0.62      0.66      0.56     79033
weighted avg       0.76      0.57      0.59     79033

Random Forest ROC-AUC Score: 0.7177

 Top 10 Most Important Features 
                                 Feature  Importance
23       CRASH_TYPE_Parked motor vehicle    0.198692
13                 CRASH_TYPE_Backing up    0.104546
35  CRASH_TYPE_Sideswipe; same direction    0.076523
25                 CRASH_TYPE_Pedestrian    0.072678
0                               LATITUDE    0.069116
1                              LONGITUDE    0.063330
10                                  pres    0.054905
6                                   temp    0.048123
7                                   rhum    0.046549
3                                 